# Part C — Enron Manager Detection and Local LLM Analysis

This notebook implements Part C using the selections in `docs/part_c_instructions.md`:

- Enron graph file: `data/raw/enron/email-Enron.txt.gz`
- Enron email content archive: `data/raw/enron/enron_mail_20150507.tar.gz`
- Manager labels: `NONE`
- Exactly three centrality algorithms: PageRank, Betweenness Centrality, and HITS Authority Score
- Local LLM tool/model: Ollama with `qwen2.5:7b-instruct`
- LLM users/emails: 10 users, 15 emails per user, max 4000 characters per email

Important limitation: the SNAP graph file uses numeric node IDs, while the CMU Enron mail archive uses mailbox folders and email addresses. The local files do not include a reliable mapping from SNAP numeric node IDs to maildir users. I therefore keep the topology-based centrality analysis and the email-content LLM analysis as related but separately identified Enron analyses, and I explain that limitation explicitly.


## Setup

I import the libraries, define paths, set deterministic sampling, and create folders for processed caches and figures. The notebook does not use external LLM APIs. The only LLM path is through a local `ollama` command if it exists and the selected model is installed.


In [ ]:
from pathlib import Path
import email
from email import policy
import gzip
import io
import json
import math
import re
import shutil
import subprocess
import tarfile
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "exports" / "figures"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENRON_GRAPH_PATH = DATA_RAW / "enron" / "email-Enron.txt.gz"
ENRON_MAIL_ARCHIVE = DATA_RAW / "enron" / "enron_mail_20150507.tar.gz"
MANAGER_LABELS_PATH = None
SOURCE_URL = "https://snap.stanford.edu/data/email-Enron.html"
LOCAL_LLM_TOOL = "Ollama"
LOCAL_LLM_MODEL = "qwen2.5:7b-instruct"
N_LLM_USERS = 10
N_EMAILS_PER_USER = 15
MAX_CHARS_PER_EMAIL = 4000
BETWEENNESS_SAMPLE_K = 250

for name, path in {"Enron graph": ENRON_GRAPH_PATH, "Enron mail archive": ENRON_MAIL_ARCHIVE}.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
print(f"Project root: {PROJECT_ROOT}")
print(f"Graph file: {ENRON_GRAPH_PATH.relative_to(PROJECT_ROOT)} ({ENRON_GRAPH_PATH.stat().st_size / 1024:.1f} KB compressed)")
print(f"Mail archive: {ENRON_MAIL_ARCHIVE.relative_to(PROJECT_ROOT)} ({ENRON_MAIL_ARCHIVE.stat().st_size / 1024**2:.1f} MB compressed)")


## C1.1 Network and Label Description

**Requirement.** Describe the Enron network, what nodes and edges represent, whether the graph is directed/weighted, available manager labels, and preprocessing.

**Method.** I load the SNAP Enron graph from the compressed edge list. The file comments say this is an email exchange network. The edge list contains directed rows, so I load it as a directed graph. If duplicate edges appear, NetworkX keeps one directed edge, so the topology graph is unweighted.

**Why this method.** The assignment asks for centrality-based manager detection, and the SNAP graph is the provided topology file. Loading it directly preserves the node IDs and directed edge orientation supplied in the dataset.


In [ ]:
def load_snap_enron_graph(path: Path) -> nx.DiGraph:
    graph = nx.DiGraph()
    comments = []
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("#"):
                comments.append(line)
                continue
            src, dst = line.split()[:2]
            graph.add_edge(int(src), int(dst))
    graph.remove_edges_from(nx.selfloop_edges(graph))
    return graph, comments

G, graph_comments = load_snap_enron_graph(ENRON_GRAPH_PATH)
UG = G.to_undirected()

label_status = "No manager label file was selected: ENRON_MANAGER_LABELS_PATH_IF_AVAILABLE = NONE"
dataset_description = pd.DataFrame([
    {
        "dataset": "SNAP Email-Enron topology graph",
        "source": SOURCE_URL,
        "local_file": str(ENRON_GRAPH_PATH.relative_to(PROJECT_ROOT)),
        "nodes": G.number_of_nodes(),
        "directed_edges": G.number_of_edges(),
        "directed": G.is_directed(),
        "weighted": False,
        "manager_labels_available": False,
        "email_content_archive": str(ENRON_MAIL_ARCHIVE.relative_to(PROJECT_ROOT)),
    }
])
display(dataset_description)
print("Graph file comments:")
for line in graph_comments[:4]:
    print(line)
print(label_status)
print(f"Weakly connected components: {nx.number_weakly_connected_components(G):,}")
print(f"Strongly connected components: {nx.number_strongly_connected_components(G):,}")


**Interpretation.** The graph is large enough for meaningful centrality rankings, but manager-label evaluation is not available because the student selected `NONE` for manager labels. The topology file also does not identify email addresses or mailbox names for numeric node IDs, which affects how the centrality and LLM sections can be connected.

### How I solved this task

I read the compressed SNAP edge list into a directed NetworkX graph, removed self-loops if any existed, counted nodes and edges, and checked the availability of labels. I also kept the raw file comments visible so the dataset definition is traceable.

**Limitations.** No precision@10 can be computed without manager labels. The graph node IDs cannot be directly matched to email bodies from the maildir archive using only the local files.


## C1.2 Centrality-Based Manager Detection

**Requirement.** Apply exactly three selected centrality algorithms, rank nodes, show the top 10 for each, and compute precision@10 if manager labels are available.

**Method.** I use exactly the selected algorithms:

1. **PageRank** on the directed graph.
2. **Betweenness Centrality** on the undirected version, approximated with `k=250` sampled source nodes because exact betweenness on 36,692 nodes is too expensive for an executed notebook.
3. **HITS Authority Score** on the directed graph.

**Why this method.** PageRank highlights nodes receiving links from important nodes, betweenness highlights bridge nodes, and HITS authority highlights nodes that are pointed to by good hub nodes. These are plausible structural signals for managerial or coordination roles in an email network.


In [ ]:
centrality_cache = DATA_PROCESSED / "part_c_enron_centrality_scores.parquet"
if centrality_cache.exists():
    centrality_df = pd.read_parquet(centrality_cache)
    centrality_status = "loaded cached centrality scores"
else:
    pagerank_scores = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-6)
    betweenness_scores = nx.betweenness_centrality(UG, k=BETWEENNESS_SAMPLE_K, seed=RANDOM_SEED, normalized=True)
    hubs, authority_scores = nx.hits(G, max_iter=100, tol=1e-8, normalized=True)
    centrality_df = pd.DataFrame({
        "node_id": list(G.nodes()),
        "pagerank": [pagerank_scores.get(node, 0.0) for node in G.nodes()],
        "betweenness_approx": [betweenness_scores.get(node, 0.0) for node in G.nodes()],
        "hits_authority": [authority_scores.get(node, 0.0) for node in G.nodes()],
        "in_degree": [G.in_degree(node) for node in G.nodes()],
        "out_degree": [G.out_degree(node) for node in G.nodes()],
        "total_degree": [G.degree(node) for node in G.nodes()],
    })
    centrality_df.to_parquet(centrality_cache, index=False)
    centrality_status = "computed and cached centrality scores"

algorithm_map = {
    "PageRank": "pagerank",
    "Betweenness Centrality": "betweenness_approx",
    "HITS Authority Score": "hits_authority",
}

top10_tables = {}
for algorithm, column in algorithm_map.items():
    top10_tables[algorithm] = centrality_df.sort_values(column, ascending=False).head(10).reset_index(drop=True)
    print(f"\nTop 10 by {algorithm}")
    display(top10_tables[algorithm][["node_id", column, "in_degree", "out_degree", "total_degree"]])

precision_at_10_status = pd.DataFrame([
    {
        "labels_available": False,
        "precision_at_10_computed": False,
        "reason": "ENRON_MANAGER_LABELS_PATH_IF_AVAILABLE = NONE, so no true manager set exists for scoring.",
        "alternative_evidence": "Compare centrality rankings and use local email evidence for separate maildir users.",
    }
])
display(precision_at_10_status)
print(f"Centrality status: {centrality_status}")
print(f"Betweenness note: approximate betweenness used k={BETWEENNESS_SAMPLE_K} sampled source nodes.")


**Interpretation.** The three algorithms produce different candidate lists because they define importance differently. PageRank and HITS authority emphasize receiving attention from important senders, while betweenness emphasizes nodes that connect otherwise distant parts of the graph. Without labels, these are candidate manager lists rather than verified manager detections.

### How I solved this task

I computed PageRank, approximate Betweenness Centrality, and HITS Authority Score, then printed the top 10 nodes for each. I did not compute precision@10 because no manager label file is available.

**Limitations.** Betweenness is approximate for runtime reasons. More importantly, numeric node IDs cannot be interpreted as names or email addresses from the local graph file alone.


## C1.3 Local LLM-Based Manager Identification

**Requirement.** Use only a local LLM, verify Ollama/model availability first, sample and truncate emails, include exact prompts, and classify likely managers from email evidence.

**Method.** I inspect the local maildir archive and select 10 mailbox users with the strongest communication evidence based on message count and unique correspondents. For each selected user, I sample up to 15 sent/received evidence emails and truncate each email to 4000 characters. Then I build the required structured prompt.

**Why this method.** The SNAP graph does not map numeric node IDs to maildir users, so I cannot honestly claim that centrality node `5038` is a specific mailbox. The LLM analysis therefore uses mailbox-level email evidence from the Enron mail archive and documents the mapping limitation.


In [ ]:
def get_text_from_message(msg) -> str:
    try:
        body = msg.get_body(preferencelist=("plain",))
        if body is not None:
            return body.get_content()
    except Exception:
        pass
    payload = msg.get_payload()
    if isinstance(payload, list):
        return "\n".join(str(part.get_payload()) for part in payload[:3])
    return str(payload)

def split_addresses(value: str) -> list[str]:
    if not value:
        return []
    # Good enough for ranking; exact RFC parsing is less important than consistent evidence sampling.
    parts = re.split(r"[,;]\s*", value.replace("\n", " "))
    return [part.strip() for part in parts if "@" in part or part.strip()]

def parse_maildir_metadata(max_files=None) -> pd.DataFrame:
    rows = []
    with tarfile.open(ENRON_MAIL_ARCHIVE, "r:gz") as tar:
        for idx, member in enumerate(tar):
            if max_files is not None and idx >= max_files:
                break
            if not member.isfile():
                continue
            parts = member.name.split("/")
            if len(parts) < 3 or parts[0] != "maildir":
                continue
            mailbox_user = parts[1]
            folder = parts[2]
            try:
                msg = email.message_from_binary_file(tar.extractfile(member), policy=policy.default)
                from_value = str(msg.get("From", ""))
                to_value = str(msg.get("To", ""))
                cc_value = str(msg.get("Cc", ""))
                subject = str(msg.get("Subject", ""))[:300]
                date_value = str(msg.get("Date", ""))
                body = get_text_from_message(msg)
                rows.append({
                    "mailbox_user": mailbox_user,
                    "folder": folder,
                    "path": member.name,
                    "from": from_value,
                    "to": to_value,
                    "cc": cc_value,
                    "subject": subject,
                    "date": date_value,
                    "body_preview": body[:MAX_CHARS_PER_EMAIL],
                    "body_length": len(body),
                })
            except Exception:
                continue
    return pd.DataFrame(rows)

metadata_cache = DATA_PROCESSED / "part_c_enron_mail_metadata.parquet"
if metadata_cache.exists():
    mail_df = pd.read_parquet(metadata_cache)
    mail_status = "loaded cached parsed mail metadata"
else:
    mail_df = parse_maildir_metadata()
    mail_df.to_parquet(metadata_cache, index=False)
    mail_status = "parsed mail archive and cached metadata"

mail_df["all_correspondents"] = (mail_df["from"].fillna("") + "; " + mail_df["to"].fillna("") + "; " + mail_df["cc"].fillna(""))
user_stats = []
for user, group in mail_df.groupby("mailbox_user"):
    correspondents = set()
    for value in group["all_correspondents"]:
        correspondents.update(split_addresses(value))
    user_stats.append({
        "mailbox_user": user,
        "email_count": len(group),
        "unique_correspondents_approx": len(correspondents),
        "sent_folder_count": int(group["folder"].str.contains("sent", case=False, na=False).sum()),
        "score_for_llm_selection": len(group) + 2 * len(correspondents),
    })
user_stats_df = pd.DataFrame(user_stats).sort_values("score_for_llm_selection", ascending=False).reset_index(drop=True)
llm_users = user_stats_df.head(N_LLM_USERS)["mailbox_user"].tolist()

def select_evidence_emails(user: str) -> pd.DataFrame:
    group = mail_df[mail_df["mailbox_user"].eq(user)].copy()
    group["sent_like"] = group["folder"].str.contains("sent", case=False, na=False)
    sent = group[group["sent_like"]].head(max(1, N_EMAILS_PER_USER // 2))
    other = group[~group["sent_like"]].head(N_EMAILS_PER_USER - len(sent))
    selected = pd.concat([sent, other], ignore_index=True)
    if len(selected) < N_EMAILS_PER_USER:
        remaining = group.drop(selected.index, errors="ignore").head(N_EMAILS_PER_USER - len(selected))
        selected = pd.concat([selected, remaining], ignore_index=True)
    return selected.head(N_EMAILS_PER_USER)

user_evidence = {user: select_evidence_emails(user) for user in llm_users}
display(user_stats_df.head(15))
print(f"Mail metadata status: {mail_status}")
print(f"Parsed mail records: {len(mail_df):,}")
print(f"Selected users for LLM evidence: {llm_users}")


In [ ]:
LLM_CLASSIFICATION_PROMPT_TEMPLATE = """You are analyzing historical Enron email evidence locally. Do not use outside knowledge.

Task: Decide whether the email user appears to be a manager or non-manager based only on the email evidence below.

User identifier: {user_id}

Evidence emails:
{email_snippets}

Return a compact structured answer:
- classification: manager / non-manager / uncertain
- confidence: low / medium / high
- reasoning: 3-5 bullet points grounded only in the emails
- evidence: quote or summarize 2-3 short email-based signals
"""

ROLE_SUMMARY_PROMPT_TEMPLATE = """You are analyzing historical Enron email evidence locally. Do not use outside knowledge.

Task: Summarize the likely workplace role of this user based only on the email evidence below.

User identifier: {user_id}

Evidence emails:
{email_snippets}

Return a compact structured answer:
- likely_role_summary: 2-4 sentences
- manager_likelihood: manager / non-manager / uncertain
- evidence_examples: 2-3 short email-based examples
- uncertainty: explain what cannot be concluded from the evidence
"""

def format_email_snippets(evidence_df: pd.DataFrame) -> str:
    snippets = []
    for idx, row in enumerate(evidence_df.itertuples(index=False), start=1):
        body = str(row.body_preview or "")[:MAX_CHARS_PER_EMAIL]
        snippets.append(
            f"Email {idx}\n"
            f"Path: {row.path}\n"
            f"Folder: {row.folder}\n"
            f"From: {row._4}\n"
            f"To: {row.to}\n"
            f"Cc: {row.cc}\n"
            f"Subject: {row.subject}\n"
            f"Date: {row.date}\n"
            f"Body excerpt (truncated to {MAX_CHARS_PER_EMAIL} chars):\n{body}"
        )
    return "\n\n---\n\n".join(snippets)

llm_prompts = []
for user in llm_users:
    snippets = format_email_snippets(user_evidence[user])
    llm_prompts.append({
        "mailbox_user": user,
        "emails_used": len(user_evidence[user]),
        "classification_prompt": LLM_CLASSIFICATION_PROMPT_TEMPLATE.format(user_id=user, email_snippets=snippets),
        "role_summary_prompt": ROLE_SUMMARY_PROMPT_TEMPLATE.format(user_id=user, email_snippets=snippets),
    })
llm_prompt_df = pd.DataFrame(llm_prompts)
display(llm_prompt_df[["mailbox_user", "emails_used"]])
print("Exact classification prompt for the first selected user:")
print(llm_prompt_df.iloc[0]["classification_prompt"][:6000])


In [ ]:
def check_ollama_model(model_name: str) -> dict:
    binary = shutil.which("ollama")
    if binary is None:
        return {
            "available": False,
            "reason": "ollama command was not found on PATH",
            "setup_commands": [
                "Install Ollama from https://ollama.com/download or your OS package manager.",
                f"ollama pull {model_name}",
                "Re-run this notebook after `ollama list` shows the model.",
            ],
        }
    try:
        result = subprocess.run([binary, "list"], capture_output=True, text=True, timeout=15)
    except Exception as exc:
        return {"available": False, "reason": f"failed to run ollama list: {exc}", "setup_commands": [f"ollama pull {model_name}"]}
    if result.returncode != 0:
        return {"available": False, "reason": result.stderr.strip() or "ollama list failed", "setup_commands": [f"ollama pull {model_name}"]}
    installed = model_name in result.stdout
    return {
        "available": installed,
        "reason": "model is installed" if installed else f"model {model_name!r} not listed by ollama",
        "ollama_list_output": result.stdout,
        "setup_commands": [] if installed else [f"ollama pull {model_name}"],
    }

def call_ollama(prompt: str, model_name: str) -> str:
    result = subprocess.run(
        ["ollama", "run", model_name],
        input=prompt,
        capture_output=True,
        text=True,
        timeout=180,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result.stdout.strip()

def parse_classification(text: str) -> str:
    lowered = text.lower()
    if "classification:" in lowered:
        after = lowered.split("classification:", 1)[1].splitlines()[0]
        if "non-manager" in after:
            return "non-manager"
        if "manager" in after:
            return "manager"
        if "uncertain" in after:
            return "uncertain"
    if "non-manager" in lowered:
        return "non-manager"
    if "manager" in lowered:
        return "manager"
    return "uncertain"

ollama_status = check_ollama_model(LOCAL_LLM_MODEL)
display(pd.DataFrame([ollama_status]))

llm_cache = DATA_PROCESSED / "part_c_llm_outputs.json"
llm_outputs = []
if ollama_status["available"]:
    for row in llm_prompt_df.itertuples(index=False):
        classification_response = call_ollama(row.classification_prompt, LOCAL_LLM_MODEL)
        role_response = call_ollama(row.role_summary_prompt, LOCAL_LLM_MODEL)
        llm_outputs.append({
            "mailbox_user": row.mailbox_user,
            "emails_used": int(row.emails_used),
            "classification_prompt": row.classification_prompt,
            "classification_response": classification_response,
            "parsed_classification": parse_classification(classification_response),
            "role_summary_prompt": row.role_summary_prompt,
            "role_summary_response": role_response,
        })
    llm_cache.write_text(json.dumps(llm_outputs, indent=2))
else:
    for row in llm_prompt_df.itertuples(index=False):
        llm_outputs.append({
            "mailbox_user": row.mailbox_user,
            "emails_used": int(row.emails_used),
            "classification_prompt": row.classification_prompt,
            "classification_response": "NOT_RUN_LOCAL_OLLAMA_UNAVAILABLE",
            "parsed_classification": "not-run",
            "role_summary_prompt": row.role_summary_prompt,
            "role_summary_response": "NOT_RUN_LOCAL_OLLAMA_UNAVAILABLE",
        })
    print("Local LLM calls were not attempted because Ollama/model availability check failed.")
    print("Setup commands:")
    for command in ollama_status.get("setup_commands", []):
        print("-", command)

llm_results_df = pd.DataFrame(llm_outputs)
display(llm_results_df[["mailbox_user", "emails_used", "parsed_classification", "classification_response"]])


**Interpretation.** The notebook prepared exact prompts and email evidence for 10 mailbox users. On this machine, the local Ollama command is unavailable, so the notebook did not run LLM inference and did not call any external model. This is the safe behavior required by the assignment when the selected local LLM is missing.

### How I solved this task

I parsed the local Enron mail archive, selected high-evidence mailbox users, sampled and truncated emails, built the exact manager-classification prompts, checked for Ollama and the selected model, and skipped inference when the local tool was unavailable.

**Limitations.** The LLM classification and role-summary outputs require installing Ollama and pulling `qwen2.5:7b-instruct`, then rerunning the notebook. The current executed notebook includes prompts and setup instructions but no completed LLM classifications because local inference was not available.


## C1.4 Summarizing Manager Roles with a Local LLM

**Requirement.** For users identified as likely managers, ask the local LLM to summarize likely role using only email evidence and include uncertainty.

**Method.** If local LLM classifications are available, select users classified as `manager`. If none are available, show the prepared role-summary prompts for the same 10 candidate users and record that role summaries were blocked by local Ollama availability.

**Why this method.** Role summaries should only be generated after local classification evidence exists. Since no external LLM may be used, the notebook must not substitute a remote model or invented summaries.


In [ ]:
manager_like_users = llm_results_df[llm_results_df["parsed_classification"].eq("manager")]["mailbox_user"].tolist()
role_summary_status = {
    "local_llm_available": bool(ollama_status["available"]),
    "manager_like_users_from_llm": manager_like_users,
    "role_summaries_completed": bool(manager_like_users),
    "note": "Role summaries are available in role_summary_response when Ollama is installed; otherwise prompts are prepared but not run.",
}
display(pd.DataFrame([role_summary_status]))
display(llm_results_df[["mailbox_user", "parsed_classification", "role_summary_response"]].head(10))
print("Exact role-summary prompt for the first selected user:")
print(llm_prompt_df.iloc[0]["role_summary_prompt"][:6000])


**Interpretation.** Role summarization is intentionally tied to local LLM availability. Because Ollama is missing here, the notebook preserves the prepared prompts and records that the summaries are not run.

### How I solved this task

I defined a separate role-summary prompt grounded only in the sampled email evidence, checked which users were classified as managers, and prepared/displayed role-summary prompts. I did not fabricate summaries when the local model was unavailable.

**Limitations.** Without local LLM execution, this section is a reproducible setup and audit trail rather than completed role classification.


## C1.5 Network Visualization

**Requirement.** Visualize the Enron network or a meaningful subgraph, size nodes by selected centrality, highlight likely managers, and explain the subgraph sampling.

**Method.** The selected visualization centrality is PageRank. I build a readable subgraph from the top PageRank nodes and their strongest neighbors by degree. Since no LLM manager classifications are available in this run, manager highlighting is based on structural candidate status: top PageRank nodes are highlighted.

**Why this method.** The full graph has tens of thousands of nodes, so a full drawing would be unreadable. A PageRank-centered subgraph keeps the visualization focused on candidate manager/coordinator nodes.


In [ ]:
TOP_PAGERANK_FOR_VIS = 20
NEIGHBORS_PER_TOP_NODE = 8
pagerank_column = "pagerank"
top_pagerank_nodes = centrality_df.sort_values(pagerank_column, ascending=False).head(TOP_PAGERANK_FOR_VIS)["node_id"].tolist()
subgraph_nodes = set(top_pagerank_nodes)
for node in top_pagerank_nodes:
    neighbors = set(G.predecessors(node)) | set(G.successors(node))
    ranked_neighbors = sorted(neighbors, key=lambda n: G.degree(n), reverse=True)[:NEIGHBORS_PER_TOP_NODE]
    subgraph_nodes.update(ranked_neighbors)
vis_graph = G.subgraph(subgraph_nodes).copy()
vis_scores = centrality_df.set_index("node_id")[pagerank_column].to_dict()
node_sizes = [250 + 7000 * math.sqrt(vis_scores.get(node, 0.0)) for node in vis_graph.nodes()]
node_colors = ["#d62728" if node in top_pagerank_nodes else "#4c78a8" for node in vis_graph.nodes()]

plt.figure(figsize=(13, 10))
pos = nx.spring_layout(vis_graph.to_undirected(), seed=RANDOM_SEED, k=0.55)
nx.draw_networkx_edges(vis_graph, pos, alpha=0.18, arrows=True, arrowsize=8, width=0.7)
nx.draw_networkx_nodes(vis_graph, pos, node_size=node_sizes, node_color=node_colors, alpha=0.88)
labels = {node: str(node) for node in top_pagerank_nodes if node in vis_graph}
nx.draw_networkx_labels(vis_graph, pos, labels=labels, font_size=8)
plt.title("Enron PageRank-centered subgraph (red = top PageRank structural candidates)")
plt.axis("off")
vis_path = FIGURES_DIR / "part_c_enron_pagerank_subgraph.png"
plt.tight_layout()
plt.savefig(vis_path, dpi=180, bbox_inches="tight")
plt.show()

vis_summary = pd.DataFrame([
    {
        "full_graph_nodes": G.number_of_nodes(),
        "full_graph_edges": G.number_of_edges(),
        "visualized_nodes": vis_graph.number_of_nodes(),
        "visualized_edges": vis_graph.number_of_edges(),
        "top_pagerank_nodes_seeded": TOP_PAGERANK_FOR_VIS,
        "neighbors_per_top_node": NEIGHBORS_PER_TOP_NODE,
        "figure_path": str(vis_path.relative_to(PROJECT_ROOT)),
    }
])
display(vis_summary)


**Interpretation.** The visualization shows a PageRank-centered part of the Enron communication graph. Red nodes are structural manager candidates by PageRank, while blue nodes are nearby high-degree communication neighbors. This is a topology visualization, not proof of job title.

### How I solved this task

I selected the top PageRank nodes, added a fixed number of their strongest neighbors, drew the resulting subgraph, scaled node size by PageRank, and highlighted top PageRank nodes in red.

**Limitations.** The graph is anonymized numeric topology, so the visualization cannot show names or email addresses. The highlighted nodes are centrality candidates, not verified managers.


## C1.6 Discussion

**Requirement.** Discuss managers identified by centrality, managers identified by local LLM, labels if available, disagreements, mistakes, and limitations.

**Method.** I compare the three centrality rankings and the local LLM status. Since manager labels and local Ollama execution are unavailable, I focus on what can and cannot be concluded from the executed notebook.

**Why this method.** A useful manager-detection notebook should not overclaim. Here, the strongest completed evidence is topology centrality; the email-content LLM workflow is prepared but blocked by missing local inference tooling.


In [ ]:
centrality_top10_summary = []
for algorithm, table in top10_tables.items():
    centrality_top10_summary.append({
        "algorithm": algorithm,
        "top10_node_ids": table["node_id"].tolist(),
    })
centrality_top10_summary_df = pd.DataFrame(centrality_top10_summary)
llm_status_summary = pd.DataFrame([
    {
        "local_llm_tool": LOCAL_LLM_TOOL,
        "local_llm_model": LOCAL_LLM_MODEL,
        "ollama_available": bool(ollama_status["available"]),
        "users_prepared_for_llm": llm_users,
        "llm_classifications_completed": bool(ollama_status["available"]),
        "manager_labels_available": False,
        "precision_at_10_status": "not computed: no manager labels selected",
    }
])
display(centrality_top10_summary_df)
display(llm_status_summary)

summary = {
    "enron_files_used": [str(ENRON_GRAPH_PATH.relative_to(PROJECT_ROOT)), str(ENRON_MAIL_ARCHIVE.relative_to(PROJECT_ROOT))],
    "centrality_algorithms_used": list(algorithm_map.keys()),
    "top10_results": {algorithm: table["node_id"].tolist() for algorithm, table in top10_tables.items()},
    "labels_available": False,
    "precision_at_10_status": "not computed because ENRON_MANAGER_LABELS_PATH_IF_AVAILABLE = NONE",
    "local_llm_tool": LOCAL_LLM_TOOL,
    "local_llm_model": LOCAL_LLM_MODEL,
    "ollama_available": bool(ollama_status["available"]),
    "llm_users_analyzed_or_prepared": llm_users,
    "llm_outputs_cache": str((DATA_PROCESSED / "part_c_llm_outputs.json").relative_to(PROJECT_ROOT)) if (DATA_PROCESSED / "part_c_llm_outputs.json").exists() else "not written because Ollama unavailable",
    "figure": str(vis_path.relative_to(PROJECT_ROOT)),
}
summary_path = DATA_PROCESSED / "part_c_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print(f"Summary written to {summary_path.relative_to(PROJECT_ROOT)}")


**Interpretation.** PageRank, approximate betweenness, and HITS authority produce structural candidate lists. Because labels are unavailable, there is no precision@10. Because Ollama is unavailable, email-content classifications and role summaries are not completed in this execution, but the notebook includes the exact local prompts and sampled evidence needed to run them after installing the selected local model.

### How I solved this task

I collected the top-10 results from each centrality algorithm, recorded the label and LLM availability status, saved a compact summary JSON, and stated the main limitations clearly.

**Limitations.** The largest limitation is the missing identity mapping between SNAP numeric nodes and maildir users. The second is missing local Ollama, which blocks the LLM-required parts until the local model is installed.
